# Basic Subsample Table Inventory
Confirm that the number of rows per partition is sensible.

In [2]:
# Prep the environment

project="measurement-lab"

import os
import collections
import pandas as pd
import matplotlib.pyplot as plt

# Set project explicitly in the environment to suppress some warnings.
os.environ["GOOGLE_CLOUD_PROJECT"] = project


In [3]:
# Depends on: pip install --upgrade google-cloud-bigquery
from google.cloud import bigquery

def run_query(query, **kwargs):
    global project, rawQuery, numberedQuery
    client = bigquery.Client(project)

    # publish rawQuery and numberedQuery to help with diagnsis
    rawQuery=query.format(**kwargs)
    numberedQuery=[]
    for n, l in enumerate(rawQuery.splitlines(), start=1):
        numberedQuery.append(f"{n} {l}".format(n, l))
    numberedQuery = "\n".join(numberedQuery)
        
    job = client.query(rawQuery)

    results = collections.defaultdict(list)
    for row in job.result():
        for key in row.keys():
            results[key].append(row.get(key))

    return pd.DataFrame(results)

In [13]:
query="""
# Generalized subsampled row inventory
# 1/16 subsample rate is hardcoded

WITH

baseData AS (
  SELECT date, COUNT(*) AS Tests
  FROM `{src}`
  WHERE date >= "{startDate}"
  GROUP BY date
),
sampledData AS (
  SELECT date, COUNT(*) AS Sampled
  FROM `{dst}`
  WHERE date >= "{startDate}"
  GROUP BY date
),
report AS (
  SELECT *,
    IFNULL(SAFE_DIVIDE(Tests/16 - Sampled,Sampled), 1.0) AS netError
  FROM baseData
    LEFT JOIN sampledData USING (date)
  ORDER BY ABS(netError) DESC, date DESC
)

SELECT * FROM report

"""
# Test code
# run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.scamper1", dst="mlab-collaboration.mm_preproduction.scamper1_DS16")

In [14]:
# scamper1
# run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.scamper1", dst="mlab-collaboration.mm_preproduction.scamper1_DS16")
# scampepr2
# run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.scamper2", dst="mlab-collaboration.mm_preproduction.scamper2_DS16")
# NDT7
run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.ndt7", dst="mlab-collaboration.mm_preproduction.ndt7_DS16")
# autoload
# run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.ndt7_dynamic", dst="mlab-collaboration.mm_preproduction.autoload_DS16")
# unified_downloads
# run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.unified_downloads", dst="mlab-collaboration.mm_preproduction.extended_intermediate_downloads_DS16")
# unified_uploads
# run_query(query, startDate='2026-05-01', src="measurement-lab.ndt.unified_uploads", dst="mlab-collaboration.mm_preproduction.extended_intermediate_uploads_DS16")

,date,Tests,Sampled,netError
0,2026-05-30,6070811,NaN,1.000000
1,2026-05-05,6613338,411399.0,0.004703
2,2026-05-22,6499992,404652.0,0.003948
3,2026-05-06,6664130,415258.0,0.003010
4,2026-05-27,5942078,370306.0,0.002900
5,2026-05-14,6708943,420398.0,-0.002591
6,2026-05-19,6664331,415466.0,0.002539
7,2026-05-21,6616138,412556.0,0.002309
8,2026-05-15,6704311,418104.0,0.002189
9,2026-05-07,6531926,408994.0,-0.001830


In [11]:
# Get the last partition in an arbitrary table
src="mlab-collaboration.mm_preproduction.extended_intermediate_downloads_DS16"

query="""
SELECT MAX(date) AS Ending FROM `{src}` WHERE date >= "2026-01-01"
"""
run_query(query, src=src)

,Ending
0,2026-04-24


In [6]:
print (numberedQuery)

1 
2 # Generalized subsampled row inventory
3 # 1/16 subsample rate is hardcoded
4 
5 WITH
6 
7 baseData AS (
8   SELECT date, COUNT(*) AS Tests
9   FROM `measurement-lab.ndt.unified_downloads`
10   WHERE date >= "2026-05-01"
11   GROUP BY date
12 ),
13 sampledData AS (
14   SELECT date, COUNT(*) AS Sampled
15   FROM `mlab-collaboration.mm_preproduction.extended_intermediate_downloads_DS16`
16   WHERE date >= "2026-05-01"
17   GROUP BY date
18 ),
19 report AS (
20   SELECT *,
21     IFNULL(SAFE_DIVIDE(Tests/16 - Sampled,Sampled), 1.0) AS netError
22   FROM baseData
23     LEFT JOIN sampledData USING (date)
24   ORDER BY ABS(netError) DESC, date DESC
25 )
26 
27 SELECT * FROM report
28 
